In [1]:
from models.montecarlo import MonteCarloModel
import numpy as np
import pandas as pd
import plotly.express as px

In [2]:
option_data = pd.read_csv('results/bsm_model_prices.csv')[['Date', 'Type', 'S0', 'K', 'T', 'r', 'sigma', 'Close Price', 'LTP']]
option_data

,Date,Type,S0,K,T,r,sigma,Close Price,LTP
0,2025-11-12,PE,25875.8,26500.0,34.0,0.1,0.074745,533.05,533.05
1,2025-11-12,CE,25875.8,25900.0,34.0,0.1,0.074745,468.90,468.90
2,2025-11-12,CE,25875.8,27450.0,34.0,0.1,0.074745,21.00,20.00
3,2025-11-12,CE,25875.8,26200.0,34.0,0.1,0.074745,286.75,286.75
4,2025-11-12,CE,25875.8,25800.0,34.0,0.1,0.074745,505.50,504.80
...,...,...,...,...,...,...,...,...,...
2748,2025-12-16,PE,25860.1,24850.0,0.0,0.1,0.078564,0.10,0.05
2749,2025-12-16,CE,25860.1,24550.0,0.0,0.1,0.078564,1342.00,1342.00
2750,2025-12-16,PE,25860.1,24250.0,0.0,0.1,0.078564,0.10,0.05
2751,2025-12-16,CE,25860.1,25400.0,0.0,0.1,0.078564,471.05,460.50


In [3]:
MC_model = MonteCarloModel(volatility_approx='gbm')
test = MC_model.price_option(option_data.iloc[1]['S0'],
                             option_data.iloc[1]['K'],
                             option_data.iloc[1]['sigma'],
                             option_data.iloc[1]['r'],
                             option_data.iloc[1]['T']/252,
                             option_type=option_data.iloc[1]['Type'], 
                             n_simulations=1000, 
                             steps=10)
test

(np.float64(458.26668017367376), np.float64(16.932082852473272))

In [4]:
mc_results = [MC_model.price_option(row['S0'], \
                row['K'], \
                row['sigma'], \
                row['r'], \
                row['T']/252, \
                option_type=row['Type'], \
                n_simulations=1000, \
                steps=10) \
                for index, row in option_data.iterrows()]

In [5]:
option_data['MC Price'] = [mc_results[i][0] for i in range(len(mc_results))]
option_data['MC StdErr'] = [mc_results[i][1] for i in range(len(mc_results))]

In [6]:
option_data['Error vs Close'] = option_data['MC Price'] - option_data['Close Price']
option_data['Error vs LTP'] = option_data['MC Price'] - option_data['LTP']
option_data['Percent Error vs Close'] = option_data['Error vs Close'] / option_data['Close Price'] * 100
option_data['Percent Error vs LTP'] = option_data['Error vs LTP'] / option_data['LTP'] * 100
option_data

,Date,Type,S0,K,T,r,sigma,Close Price,LTP,MC Price,MC StdErr,Error vs Close,Error vs LTP,Percent Error vs Close,Percent Error vs LTP
0,2025-11-12,PE,25875.8,26500.0,34.0,0.1,0.074745,533.05,533.05,423.711271,1.519222e+01,-109.338729,-109.338729,-20.511909,-20.511909
1,2025-11-12,CE,25875.8,25900.0,34.0,0.1,0.074745,468.90,468.90,448.941363,1.610758e+01,-19.958637,-19.958637,-4.256480,-4.256480
2,2025-11-12,CE,25875.8,27450.0,34.0,0.1,0.074745,21.00,20.00,14.572274,2.707186e+00,-6.427726,-5.427726,-30.608218,-27.138629
3,2025-11-12,CE,25875.8,26200.0,34.0,0.1,0.074745,286.75,286.75,298.905609,1.335773e+01,12.155609,12.155609,4.239096,4.239096
4,2025-11-12,CE,25875.8,25800.0,34.0,0.1,0.074745,505.50,504.80,514.596658,1.750944e+01,9.096658,9.796658,1.799537,1.940701
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2748,2025-12-16,PE,25860.1,24850.0,0.0,0.1,0.078564,0.10,0.05,0.000000,0.000000e+00,-0.100000,-0.050000,-100.000000,-100.000000
2749,2025-12-16,CE,25860.1,24550.0,0.0,0.1,0.078564,1342.00,1342.00,1310.100000,7.193785e-15,-31.900000,-31.900000,-2.377049,-2.377049
2750,2025-12-16,PE,25860.1,24250.0,0.0,0.1,0.078564,0.10,0.05,0.000000,0.000000e+00,-0.100000,-0.050000,-100.000000,-100.000000
2751,2025-12-16,CE,25860.1,25400.0,0.0,0.1,0.078564,471.05,460.50,460.100000,0.000000e+00,-10.950000,-0.400000,-2.324594,-0.086862


In [7]:
print("Mean Error vs Close Price:", option_data['Error vs Close'].mean())
print("Mean Error vs LTP:", option_data['Error vs LTP'].mean())
print("Mean Percent Error vs Close Price:", option_data['Percent Error vs Close'].mean())
print("Mean Percent Error vs LTP:", option_data['Percent Error vs LTP'].mean())
print("Std Dev of Percent Error vs Close Price:", option_data['Percent Error vs Close'].std())
print("Std Dev of Percent Error vs LTP:", option_data['Percent Error vs LTP'].std())

Mean Error vs Close Price: -7.687396473357207
Mean Error vs LTP: -7.257519974991789
Mean Percent Error vs Close Price: -32.36119164865658
Mean Percent Error vs LTP: -32.10206447756426
Std Dev of Percent Error vs Close Price: 45.78679044200256
Std Dev of Percent Error vs LTP: 45.91515750490525


In [8]:
fig = px.histogram(option_data, x='Percent Error vs LTP', nbins=50, title='Histogram of Percent Error vs LTP (Monte Carlo Model)')
fig.show()

In [11]:
fig = px.scatter(option_data, x='LTP', y='MC Price', title='Monte Carlo Model Prices vs LTP')
fig.add_shape(type='line', x0=option_data['LTP'].min(), y0=option_data['LTP'].min(), x1=option_data['LTP'].max(), y1=option_data['LTP'].max(),
              line=dict(color='Red'))
fig.show()

In [12]:
fig = px.scatter(option_data, x='LTP', y='Percent Error vs LTP', title='Percent Error vs LTP')
fig.add_shape(type='line', x0=0, y0=0, x1=option_data['Close Price'].max(), y1=0,
              line=dict(color='Red',), xref='x', yref='y')
fig.show()

In [13]:
option_data.to_csv('results/monte_carlo_model_prices.csv', index=False)